# 07. 같은 지점 비교 분석 (case-crossover)

입력: `data/3_결과/train_table.csv` (05 노트북 결과)

**무엇을 하나**
- 사고 1건마다 **같은 자리**의 "평소" 3~4개(같은 달·같은 요일·같은 시각의 다른 주)를 묶음으로 만듦
- 묶음 안에서만 비교 → 도로 모양·터널 자체 같은 **장소 차이는 지워짐**
- 남는 것: 그날그날 달라지는 것(햇빛 → A, 비)의 효과

**결과 읽는 법**
- `A 0.1당 1.38배` = A가 0.1 오른 날은 같은 자리 평소보다 사고 오즈가 1.38배
- 95% 범위가 1을 포함하면 "아직 확실하지 않음"

**비(rain)는 왜 넣나** (목표가 아님)
1. 저울 점검: "비 → 사고↑"가 나오면 방법이 작동한다는 뜻
2. 비 오는 날은 어두워 A가 낮음 → 비를 따로 빼야 비 효과가 A에 섞이지 않음

> 초안 · 업데이트 중 (검토 전). 교통량(VDS)·길이 그룹이 들어가면 숫자는 바뀔 수 있음

In [1]:
import pandas as pd, numpy as np, warnings
from pathlib import Path
from statsmodels.discrete.conditional_models import ConditionalLogit
warnings.filterwarnings('ignore')

DATA = Path('data')
t = pd.read_csv(DATA / '3_결과' / 'train_table.csv', low_memory=False)

# 사고(1) + 같은 지점 평소(0)만, 평소가 1개 이상 있는 사고만
cc = t[t.design.isin(['사고', '같은지점'])].copy()
cc = cc[cc.case_id.isin(cc[cc.y == 0].case_id)]
cc['rain'] = (cc.rn.fillna(0) > 0).astype(int)
cc['vs_km'] = cc.vs / 100
cc['hol'] = cc.is_holiday.astype(int)
print('사고', cc[cc.y == 1].case_id.nunique(), '건 / 평소', (cc.y == 0).sum(), '행')

사고 4443 건 / 평소 15019 행


In [2]:
def fit(d, cols, label):
    """조건부 로지스틱: 사고와 그 사고의 평소끼리만 비교. 결과는 배수(오즈비)."""
    d = d.dropna(subset=cols)
    d = d[d.groupby('case_id').y.transform('sum') == 1]
    d = d[d.groupby('case_id').y.transform('size') > 1]
    m = ConditionalLogit(d.y, d[cols], groups=d.case_id).fit(disp=0)
    ci = m.conf_int()
    rows = []
    for c in cols:
        rows.append({'항목': c, '배수': round(np.exp(m.params[c]), 2),
                     '95%_하한': round(np.exp(ci.loc[c, 0]), 2),
                     '95%_상한': round(np.exp(ci.loc[c, 1]), 2),
                     'p': round(m.pvalues[c], 3)})
    print(f'== {label}: 사고 {d.case_id.nunique()}건')
    return pd.DataFrame(rows)

## 1. 저울 점검: 전체 사고에서 비·시정·공휴일

In [3]:
fit(cc, ['rain', 'vs_km', 'hol'], '전체 (방법 점검)')

== 전체 (방법 점검): 사고 4380건


,항목,배수,95%_하한,95%_상한,p
0,rain,1.78,1.57,2.02,0.000
1,vs_km,1.00,1.00,1.00,0.256
2,hol,0.91,0.77,1.07,0.247


## 2. 구간별 사고 수 (낮 = 태양 고도 > 0)

In [4]:
order = ['진입전', '경계부', '이행부', '완화부', '기본부', '출구부', '진출후', '일반']
case = cc[cc.y == 1]
pd.crosstab(case.tunnel_zone, np.where(case.sun_elev > 0, '낮', '밤')).reindex(order)

col_0,낮,밤
tunnel_zone,,
진입전,83,29
경계부,42,14
이행부,28,6
완화부,66,10
기본부,147,55
출구부,32,5
진출후,39,21
일반,2389,1477


## 3. 핵심: A가 오르면 사고 위험이 몇 배?

- 진입 A: 진입전 + 경계부 구간 낮 사고
- 진출 A: 출구부 + 진출후 구간 낮 사고
- A를 10배 해서 넣음 → 결과가 "A 0.1당 배수"

In [5]:
res = {}
for zones, col, lab in [(['진입전', '경계부'], 'A_진입', '진입'),
                        (['출구부', '진출후'], 'A_진출', '진출')]:
    z = cc[cc.tunnel_zone.isin(zones)]
    day_cases = z[(z.y == 1) & (z.sun_elev > 0)].case_id
    d = z[z.case_id.isin(day_cases)].copy()
    d['A10'] = d[col] * 10
    res[lab] = fit(d, ['A10', 'rain'], f'{lab} 낮, A 0.1당')
    display(res[lab])

== 진입 낮, A 0.1당: 사고 125건


,항목,배수,95%_하한,95%_상한,p
0,A10,1.29,0.94,1.78,0.112
1,rain,1.27,0.44,3.66,0.652


== 진출 낮, A 0.1당: 사고 71건


,항목,배수,95%_하한,95%_상한,p
0,A10,0.95,0.78,1.16,0.617
1,rain,3.16,1.14,8.76,0.027


## 4. 발표용: A를 3등급으로 나눠 비교 (진입)

| 등급 | 밝기 차이(바깥÷터널 안) | A |
|---|---|---|
| 낮음 | 10배 미만 | 0.50 미만 |
| 보통 | 10~15배 | 0.50~0.59 |
| 높음 | 15배 이상 | 0.59 이상 |

기준 = 낮음. "보통·높음 날은 낮음 날보다 몇 배"

In [6]:
z = cc[cc.tunnel_zone.isin(['진입전', '경계부'])]
day_cases = z[(z.y == 1) & (z.sun_elev > 0)].case_id
d = z[z.case_id.isin(day_cases)].copy()
d['등급'] = pd.cut(d['A_진입'], [-1, 0.5, np.log10(15) / 2, 2], right=False,
                 labels=['낮음', '보통', '높음'])
print(pd.crosstab(d['등급'], d.y.map({1: '사고', 0: '평소'})))
x = pd.get_dummies(d, columns=['등급'], drop_first=True, dtype=int)
fit(x, ['등급_보통', '등급_높음', 'rain'], '진입 낮, 등급별 (기준 낮음)')

y   사고   평소
등급         
낮음  35  144
보통  66  209
높음  24   70
== 진입 낮, 등급별 (기준 낮음): 사고 125건


,항목,배수,95%_하한,95%_상한,p
0,등급_보통,1.57,0.88,2.79,0.123
1,등급_높음,1.95,0.85,4.49,0.114
2,rain,1.13,0.42,3.04,0.816


## 6. 천리안 일사량으로 A 다시 계산해서 비교

04b에서 받은 천리안 값(`data/2_중간/gk2a_solar.csv`)으로 A를 **05와 똑같은 식**으로 다시 계산하고, 3번 분석을 그대로 반복해요.
- 비교는 **두 값이 모두 있는 묶음만** 써서 공정하게 맞춤 (같은 사고·같은 평소)
- 두 결과의 배수 방향이 같으면 → 결론이 일사량 출처에 흔들리지 않음 (튼튼함)
- 천리안 쪽 95% 범위가 더 좁아지면 → 관측소 거리 때문에 흐려졌던 신호가 살아난 것


In [7]:
g = pd.read_csv(DATA / '2_중간' / 'gk2a_solar.csv', parse_dates=['t'])[['case_id', 'y', 't', 'gk2a_icsr']]
cc['t'] = pd.to_datetime(cc.t)
cc2 = cc.merge(g, on=['case_id', 'y', 't'], how='left')

def a_score(icsr, zone):
    """05와 같은 식: 일사량 → 바깥 밝기 L20 → 터널 안 밝기와의 비 → A (0~1)"""
    L20 = 0.15 * (icsr * 277.8 * 126) / np.pi
    step = np.clip(np.ceil(L20 / 4000 * 4) / 4, 0.25, 1)          # 조광 4단계
    s = lambda R: np.clip(np.log10(np.where(R > 0, R, 1)) / 2, 0, 1)
    a_in = np.where(zone.isin(['진입전', '경계부']), s(L20 / (280 * step)), 0.0)
    a_out = np.where(zone.isin(['출구부', '진출후']), s(L20 / 45), 0.0)
    return a_in, a_out

cc2['A_진입_천리안'], cc2['A_진출_천리안'] = a_score(cc2.gk2a_icsr, cc2.tunnel_zone)
cc2.loc[cc2.gk2a_icsr.isna(), ['A_진입_천리안', 'A_진출_천리안']] = np.nan
# 계산 확인: ASOS 일사량을 넣으면 05의 A와 같아야 함
chk_in, chk_out = a_score(cc2.icsr.fillna(0), cc2.tunnel_zone)
print('식 확인 (05와 일치):', np.allclose(chk_in, cc2.A_진입) and np.allclose(chk_out, cc2.A_진출))


식 확인 (05와 일치): True


In [8]:
out = []
for zones, lab in [(['진입전', '경계부'], '진입'), (['출구부', '진출후'], '진출')]:
    z = cc2[cc2.tunnel_zone.isin(zones)]
    day_cases = z[(z.y == 1) & (z.sun_elev > 0)].case_id
    d = z[z.case_id.isin(day_cases)].dropna(subset=[f'A_{lab}_천리안'])
    for src, col in [('ASOS', f'A_{lab}'), ('천리안', f'A_{lab}_천리안')]:
        d = d.assign(A10=d[col] * 10)
        r = fit(d, ['A10', 'rain'], f'{lab} 낮 · {src}').iloc[[0]]
        out.append(r.assign(구간=lab, 일사량=src))
pd.concat(out)[['구간', '일사량', '배수', '95%_하한', '95%_상한', 'p']]


== 진입 낮 · ASOS: 사고 123건
== 진입 낮 · 천리안: 사고 123건
== 진출 낮 · ASOS: 사고 69건
== 진출 낮 · 천리안: 사고 69건


,구간,일사량,배수,95%_하한,95%_상한,p
0,진입,ASOS,1.26,0.92,1.74,0.153
0,진입,천리안,1.16,0.86,1.57,0.331
0,진출,ASOS,0.96,0.79,1.18,0.703
0,진출,천리안,0.97,0.78,1.19,0.746


## 5. 해석 메모

- 진입: 밝을수록 위험한 방향. 범위가 1을 살짝 포함 → "경향" 수준
- 진출: 아직 신호 없음 (낮 사고 64건으로 적음)
- 다음: 교통량(VDS) 추가, 터널 길이 그룹별 비교, 같은 날 ±1~2시간 보조 분석